# Lipsync de las entrevistas · BandManager.io

Gratis, con la GPU de Google Colab y **LatentSync 1.5** (ByteDance, licencia abierta).

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar*.

Luego *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube `lipsync.zip`. Al final se descarga `resultados.zip`: pásaselo a Claude.

## 1. Comprobar que hay GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'SIN GPU: cambia el tipo de entorno a GPU T4 y vuelve a ejecutar'

## 2. Instalar LatentSync (unos 10 minutos la primera vez)
Se instala en un Python 3.10 aparte, que es el que pide LatentSync, para no chocar con el de Colab.

In [ ]:
%%bash
set -e
cd /content
[ -d LatentSync ] || git clone -q https://github.com/bytedance/LatentSync.git
pip -q install uv
[ -d /content/venv ] || uv venv -q /content/venv --python 3.10
PY=/content/venv/bin/python
uv pip install -q --python $PY numpy==1.26.4 cython setuptools wheel
uv pip install -q --python $PY --index-strategy unsafe-best-match --no-build-isolation-package insightface -r LatentSync/requirements.txt
echo 'Instalado'

## 3. Descargar el modelo LatentSync 1.5 (≈ 5 GB)

In [ ]:
from huggingface_hub import hf_hub_download
for f in ['latentsync_unet.pt', 'whisper/tiny.pt']:
    hf_hub_download('ByteDance/LatentSync-1.5', f, local_dir='/content/LatentSync/checkpoints')
print('Modelo listo')

## 4. Subir `lipsync.zip`

In [ ]:
from google.colab import files
import zipfile, os, shutil
subido = files.upload()
nombre = next(iter(subido))
shutil.rmtree('/content/entrada', ignore_errors=True)
zipfile.ZipFile(nombre).extractall('/content/entrada')
base = '/content/entrada/lipsync' if os.path.isdir('/content/entrada/lipsync') else '/content/entrada'
carpetas = sorted(d for d in os.listdir(base) if os.path.exists(os.path.join(base, d, 'video.mp4')))
print(len(carpetas), 'entrevistas:', ', '.join(carpetas))

## 5. Sincronizar los labios (1-2 minutos por entrevista)

In [ ]:
import subprocess
os.makedirs('/content/resultados', exist_ok=True)
for c in carpetas:
    d = os.path.join(base, c)
    salida = f'/content/resultados/{c}.mp4'
    if os.path.exists(salida):
        print('ya estaba:', c); continue
    wav = os.path.join(d, 'audio.wav')
    subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', os.path.join(d, 'audio.mp3'), '-ar', '16000', '-ac', '1', wav], check=True)
    r = subprocess.run(['/content/venv/bin/python', '-m', 'scripts.inference',
                        '--unet_config_path', 'configs/unet/stage2.yaml',
                        '--inference_ckpt_path', 'checkpoints/latentsync_unet.pt',
                        '--inference_steps', '20', '--guidance_scale', '1.5',
                        '--video_path', os.path.join(d, 'video.mp4'), '--audio_path', wav,
                        '--video_out_path', salida], cwd='/content/LatentSync', capture_output=True, text=True)
    print(('OK   ' if r.returncode == 0 and os.path.exists(salida) else 'FALLO ') + c)
    if r.returncode != 0:
        print(r.stderr[-1500:])

## 6. Descargar `resultados.zip` y pasárselo a Claude

In [ ]:
shutil.make_archive('/content/resultados', 'zip', '/content/resultados')
print(sorted(os.listdir('/content/resultados')))
files.download('/content/resultados.zip')